# Metagenomic Disease Classification - Model Comparison

In this notebook, I compare several supervised learning methods for metagenomic disease classification using sample-level abundance features.

The main goal here is to test multiple classifiers on the same feature matrix and compare their cross-validated performance.

The methods I included are:
- Logistic Regression
- Random Forest
- Support Vector Machine
- Gaussian Process Classifier
- XGBoost

I also included an optional Bayesian optimization section for tuning one of the models afterward.

## Overview

The general workflow is:

1. Start with a sample-by-feature matrix
2. Separate features from labels
3. Train several supervised models
4. Compare them with cross-validation
5. Save the best model

This notebook uses a mock example dataset for demonstration, but the same structure works with a real metagenomic abundance table.

In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF

warnings.filterwarnings("ignore")

## Optional XGBoost import

If `xgboost` is installed, this cell will add it to the comparison. Otherwise the rest of the notebook still works.

In [ ]:
HAS_XGBOOST = True
try:
    from xgboost import XGBClassifier
except ImportError:
    HAS_XGBOOST = False
    print("xgboost is not installed. The notebook will skip the XGBoost model.")

HAS_XGBOOST

## Optional Bayesian optimization import

This section is optional too. If `optuna` is installed, I can use it later for tuning.

In [ ]:
HAS_OPTUNA = True
try:
    import optuna
    from sklearn.model_selection import cross_val_score
except ImportError:
    HAS_OPTUNA = False
    print("optuna is not installed. The notebook will skip Bayesian optimization.")

HAS_OPTUNA

## Example feature table

This is a small mock metagenomic abundance table for presentation purposes.

In a real setting, I would replace this with the actual sample-by-taxon matrix produced from Kraken 2 and Bracken outputs.

In [ ]:
feature_df = pd.DataFrame([
    {"sample_id": "sample_001", "label": "diseaseA", "Bacteroides_fragilis": 0.20, "Escherichia_coli": 0.05, "Faecalibacterium_prausnitzii": 0.01, "Prevotella_copri": 0.02},
    {"sample_id": "sample_002", "label": "diseaseA", "Bacteroides_fragilis": 0.22, "Escherichia_coli": 0.04, "Faecalibacterium_prausnitzii": 0.02, "Prevotella_copri": 0.03},
    {"sample_id": "sample_003", "label": "diseaseA", "Bacteroides_fragilis": 0.18, "Escherichia_coli": 0.07, "Faecalibacterium_prausnitzii": 0.03, "Prevotella_copri": 0.02},
    {"sample_id": "sample_004", "label": "diseaseB", "Bacteroides_fragilis": 0.03, "Escherichia_coli": 0.28, "Faecalibacterium_prausnitzii": 0.01, "Prevotella_copri": 0.15},
    {"sample_id": "sample_005", "label": "diseaseB", "Bacteroides_fragilis": 0.04, "Escherichia_coli": 0.31, "Faecalibacterium_prausnitzii": 0.00, "Prevotella_copri": 0.12},
    {"sample_id": "sample_006", "label": "diseaseB", "Bacteroides_fragilis": 0.05, "Escherichia_coli": 0.26, "Faecalibacterium_prausnitzii": 0.01, "Prevotella_copri": 0.14},
    {"sample_id": "sample_007", "label": "healthy",  "Bacteroides_fragilis": 0.12, "Escherichia_coli": 0.02, "Faecalibacterium_prausnitzii": 0.25, "Prevotella_copri": 0.03},
    {"sample_id": "sample_008", "label": "healthy",  "Bacteroides_fragilis": 0.10, "Escherichia_coli": 0.01, "Faecalibacterium_prausnitzii": 0.27, "Prevotella_copri": 0.02},
    {"sample_id": "sample_009", "label": "healthy",  "Bacteroides_fragilis": 0.11, "Escherichia_coli": 0.03, "Faecalibacterium_prausnitzii": 0.24, "Prevotella_copri": 0.01},
])

feature_df

## Separate features and labels

In [ ]:
X = feature_df.drop(columns=["sample_id", "label"])
y = feature_df["label"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:", list(label_encoder.classes_))
print("Feature matrix shape:", X.shape)

## Define the models

I kept the setup simple and consistent so the comparison is easier to interpret.

In [ ]:
models = {
    "logistic_regression": Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced")),
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("clf", RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")),
    ]),
    "svm": Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
    ]),
    "gaussian_process": Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler", StandardScaler()),
        ("clf", GaussianProcessClassifier(kernel=1.0 * RBF(length_scale=1.0), random_state=42)),
    ]),
}

if HAS_XGBOOST:
    models["xgboost"] = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("clf", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="mlogloss",
            random_state=42,
        )),
    ])

list(models.keys())

## Cross-validation comparison

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y_encoded,
        cv=cv,
        scoring=["accuracy", "f1_macro", "precision_macro", "recall_macro"],
        return_train_score=False,
    )

    results.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "f1_macro": scores["test_f1_macro"].mean(),
        "precision_macro": scores["test_precision_macro"].mean(),
        "recall_macro": scores["test_recall_macro"].mean(),
    })

results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
results_df

## Pick the best model based on macro F1

In [ ]:
best_model_name = results_df.loc[0, "model"]
best_model = models[best_model_name]

print("Best model:", best_model_name)

## Fit the best model on the full dataset and save it

In [ ]:
MODEL_DIR = Path.cwd() / "model_comparison_outputs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

best_model.fit(X, y_encoded)

joblib.dump(best_model, MODEL_DIR / "best_metagenomic_model.joblib")
joblib.dump(label_encoder, MODEL_DIR / "best_metagenomic_label_encoder.joblib")
joblib.dump(list(X.columns), MODEL_DIR / "feature_columns.joblib")

print("Saved files to:")
print(MODEL_DIR)

## Predict on a new sample

In [ ]:
new_sample_df = pd.DataFrame([
    {
        "sample_id": "new_sample_001",
        "Bacteroides_fragilis": 0.04,
        "Escherichia_coli": 0.25,
        "Faecalibacterium_prausnitzii": 0.01,
        "Prevotella_copri": 0.13,
    }
])

X_new = new_sample_df.drop(columns=["sample_id"])
X_new = X_new.reindex(columns=X.columns, fill_value=0.0)

pred = best_model.predict(X_new)
proba = best_model.predict_proba(X_new)
labels = label_encoder.inverse_transform(pred)

for sample_id, label, probs in zip(new_sample_df["sample_id"], labels, proba):
    print("sample:", sample_id)
    print("predicted_label:", label)
    print("class_probabilities:")
    for cls, p in zip(label_encoder.classes_, probs):
        print(f"  {cls}: {p:.4f}")

## Optional Bayesian optimization with Optuna

This section tunes a random forest model using Optuna. It is optional and only runs if `optuna` is installed.

In [ ]:
if HAS_OPTUNA:
    def objective(trial):
        n_estimators = trial.suggest_int("n_estimators", 100, 500)
        max_depth = trial.suggest_int("max_depth", 2, 12)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
            ("clf", RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
                class_weight="balanced",
                random_state=42,
            )),
        ])

        scores = cross_val_score(model, X, y_encoded, cv=cv, scoring="f1_macro")
        return scores.mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=25)

    print("Best Optuna score:", study.best_value)
    print("Best parameters:")
    print(study.best_params)
else:
    print("Optuna section skipped because optuna is not installed.")

## Notes on interpretation

- Logistic regression gives a strong linear baseline.
- Random forest handles nonlinear interactions without much tuning.
- SVM can work well on smaller datasets.
- Gaussian processes can be useful when I want probabilistic predictions and uncertainty, but they may become slow as the dataset grows.
- XGBoost is often a strong choice for tabular metagenomic features.
- Bayesian optimization is useful for tuning model hyperparameters, but it is not the classifier itself.

## Final summary

This notebook is meant to compare several supervised approaches for metagenomic disease classification on the same feature table.

With real data, the next steps would be:
- replace the mock feature table with real abundance features
- evaluate on a larger labeled cohort
- check for batch effects and leakage
- compare results across taxonomic levels such as species vs genus
- tune the best-performing model more carefully